## Data Cleaning Notebook

1. Cleaning the raw extracted data from airbnb stored at `"../Data/raw/Airbnbdata` to a clean csv ready for analysis and model training
2. Further cleaning after scrabbing extra columns data

In [156]:
import pandas as pd
import json
import re

with open(r"../Data/raw/Airbnb_Scrapped.json", encoding="utf-8") as f:
    data = json.load(f)
    
if isinstance(data, dict):
    data = [data]

df = pd.json_normalize(data)

df.columns = (
    df.columns
    .str.replace('.', '_')  
    .str.lower()             
)

if 'price_price' in df.columns:
    df['price_price'] = (
        df['price_price']
        .astype(str)
        .str.replace('$', '', regex=False)
        
    )

print("Shape:", df.shape)

Shape: (833, 36)


In [157]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 833 entries, 0 to 832
Data columns (total 36 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   thumbnail                                 833 non-null    object 
 1   id                                        833 non-null    object 
 2   title                                     783 non-null    object 
 3   description                               833 non-null    object 
 4   url                                       833 non-null    object 
 5   rating_accuracy                           467 non-null    float64
 6   rating_checking                           467 non-null    float64
 7   rating_cleanliness                        467 non-null    float64
 8   rating_communication                      467 non-null    float64
 9   rating_location                           467 non-null    float64
 10  rating_value                          

#### Get price from json `Price/breakdown/basePrice/description` most accurate 

In [158]:
import numpy as np
def extract_nightly_price_from_str(desc, price=None):
    """Extract nightly price from a string like '3 nights x $120.00' or fallback to price/num_nights."""
    if isinstance(desc, str):
        match = re.search(r'(\d+)\s*nights?\s*x\s*\$(\d+(?:\.\d+)?)', desc)
        if match:
            return float(match.group(2))
        # Fallback: try to divide price by nights if both are available
        nights_match = re.search(r'(\d+)\s*nights?', desc)
        if nights_match and price is not None:
            try:
                nights = int(nights_match.group(1))
                base_total = float(str(price).replace("$", "").replace(",", ""))
                if nights > 0:
                    return round(base_total / nights, 2)
            except Exception:
                pass
    return np.nan
# Apply extraction to the DataFrame
df['nightly_price'] = df.apply(lambda row: extract_nightly_price_from_str(row['price_breakdown_baseprice_description'], row.get('price_breakdown_baseprice_price')), axis=1)


#### Extract checkin and checkout data from URL

In [159]:
import urllib.parse

def extract_checkin_checkout(url):
    """Extract check-in and checkout dates from Airbnb URL if present as query params."""
    if not isinstance(url, str):
        return pd.Series([None, None])
    parsed = urllib.parse.urlparse(url)
    params = urllib.parse.parse_qs(parsed.query)
    checkin = params.get('check_in', [None])[0]
    checkout = params.get('check_out', [None])[0]
    return pd.Series([checkin, checkout])

# Apply extraction to the DataFrame
if 'url' in df.columns:
    df[['checkin_date', 'checkout_date']] = df['url'].apply(extract_checkin_checkout)
else:
    df['checkin_date'] = None
    df['checkout_date'] = None

df[['url', 'checkin_date', 'checkout_date']].head()

,url,checkin_date,checkout_date
0,https://www.airbnb.com/rooms/12927132341549453...,2026-05-31,2026-06-05
1,https://www.airbnb.com/rooms/15087185116306463...,2026-05-24,2026-05-29
2,https://www.airbnb.com/rooms/12973272196317893...,2026-06-24,2026-06-29
3,https://www.airbnb.com/rooms/16060018531994111...,2026-05-11,2026-05-16
4,https://www.airbnb.com/rooms/13148334670964898...,2026-05-03,2026-05-08


In [160]:
df['checkin_date'] = pd.to_datetime(df['checkin_date'], errors='coerce')
df['checkout_date'] = pd.to_datetime(df['checkout_date'], errors='coerce')

In [161]:
df['checkin_year'] = df['checkin_date'].dt.year
df['checkin_month'] = df['checkin_date'].dt.month
df['checkin_day'] = df['checkin_date'].dt.day
df['checkin_weekday'] = df['checkin_date'].dt.weekday

df['checkout_year'] = df['checkout_date'].dt.year
df['checkout_month'] = df['checkout_date'].dt.month
df['checkout_day'] = df['checkout_date'].dt.day
df['checkout_weekday'] = df['checkout_date'].dt.weekday

In [162]:

cols_to_keep = [
    "id", "title", "url", "thumbnail",
    "coordinates_latitude", "coordinates_longitude",
    "rating_guestsatisfaction", "rating_reviewscount",
    "rating_accuracy", "rating_cleanliness",
    "rating_value", "rating_location",
    "description", "price_breakdown_baseprice_price",
    "nightly_price", "price_price",
    "checkin_year", "checkin_month", "checkin_day", "checkin_weekday",
    "checkout_year", "checkout_month", "checkout_day", "checkout_weekday"
    
]

cols_to_keep = [c for c in cols_to_keep if c in df.columns]
df = df[cols_to_keep].copy()


df.head()

,id,title,url,thumbnail,coordinates_latitude,coordinates_longitude,rating_guestsatisfaction,rating_reviewscount,rating_accuracy,rating_cleanliness,...,nightly_price,price_price,checkin_year,checkin_month,checkin_day,checkin_weekday,checkout_year,checkout_month,checkout_day,checkout_weekday
0,1292713234154945394,"ETERNA.Suite W Jaccuzi, Pyramids View & Balcony",https://www.airbnb.com/rooms/12927132341549453...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.973873,31.146603,4.95,133.0,4.96,4.92,...,150.00,675,2026,5,31,6,2026,6,5,4
1,1508718511630646313,king khufu suite,https://www.airbnb.com/rooms/15087185116306463...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.986300,31.143100,5.00,62.0,5.00,5.00,...,105.37,527,2026,5,24,6,2026,5,29,4
2,1297327219631789358,Akasia Pyramids View,https://www.airbnb.com/rooms/12973272196317893...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.978100,31.145400,4.91,176.0,4.91,4.91,...,38.64,194,2026,6,24,2,2026,6,29,0
3,1606001853199411128,Mountain Cave | Pyramids View & Jacuzzi,https://www.airbnb.com/rooms/16060018531994111...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.979090,31.146920,4.92,12.0,5.00,4.75,...,118.49,593,2026,5,11,0,2026,5,16,5
4,1314833467096489875,Heaven of Pyramids,https://www.airbnb.com/rooms/13148334670964898...,https://a0.muscache.com/im/pictures/miso/Hosti...,29.978787,31.144191,4.71,125.0,4.74,4.61,...,23.22,117,2026,5,3,6,2026,5,8,4


In [163]:
# to rename the columns names
df.rename(columns={
    "coordinates_latitude":               "lat",
    "coordinates_longitude":              "lng",
    "rating_guestsatisfaction":           "rating_overall",
    "rating_reviewscount":                "reviews_count",    
}, inplace=True)
df.head()

,id,title,url,thumbnail,lat,lng,rating_overall,reviews_count,rating_accuracy,rating_cleanliness,...,nightly_price,price_price,checkin_year,checkin_month,checkin_day,checkin_weekday,checkout_year,checkout_month,checkout_day,checkout_weekday
0,1292713234154945394,"ETERNA.Suite W Jaccuzi, Pyramids View & Balcony",https://www.airbnb.com/rooms/12927132341549453...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.973873,31.146603,4.95,133.0,4.96,4.92,...,150.00,675,2026,5,31,6,2026,6,5,4
1,1508718511630646313,king khufu suite,https://www.airbnb.com/rooms/15087185116306463...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.986300,31.143100,5.00,62.0,5.00,5.00,...,105.37,527,2026,5,24,6,2026,5,29,4
2,1297327219631789358,Akasia Pyramids View,https://www.airbnb.com/rooms/12973272196317893...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.978100,31.145400,4.91,176.0,4.91,4.91,...,38.64,194,2026,6,24,2,2026,6,29,0
3,1606001853199411128,Mountain Cave | Pyramids View & Jacuzzi,https://www.airbnb.com/rooms/16060018531994111...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.979090,31.146920,4.92,12.0,5.00,4.75,...,118.49,593,2026,5,11,0,2026,5,16,5
4,1314833467096489875,Heaven of Pyramids,https://www.airbnb.com/rooms/13148334670964898...,https://a0.muscache.com/im/pictures/miso/Hosti...,29.978787,31.144191,4.71,125.0,4.74,4.61,...,23.22,117,2026,5,3,6,2026,5,8,4


In [164]:
print(df.isnull().sum())
print("\n")

id                                   0
title                               50
url                                  0
thumbnail                            0
lat                                  2
lng                                  2
rating_overall                     366
reviews_count                      366
rating_accuracy                    366
rating_cleanliness                 366
rating_value                       366
rating_location                    366
description                          0
price_breakdown_baseprice_price      0
nightly_price                        0
price_price                          0
checkin_year                         0
checkin_month                        0
checkin_day                          0
checkin_weekday                      0
checkout_year                        0
checkout_month                       0
checkout_day                         0
checkout_weekday                     0
dtype: int64




In [165]:
df.columns

Index(['id', 'title', 'url', 'thumbnail', 'lat', 'lng', 'rating_overall',
       'reviews_count', 'rating_accuracy', 'rating_cleanliness',
       'rating_value', 'rating_location', 'description',
       'price_breakdown_baseprice_price', 'nightly_price', 'price_price',
       'checkin_year', 'checkin_month', 'checkin_day', 'checkin_weekday',
       'checkout_year', 'checkout_month', 'checkout_day', 'checkout_weekday'],
      dtype='object')

In [166]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 833 entries, 0 to 832
Data columns (total 24 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   id                               833 non-null    object 
 1   title                            783 non-null    object 
 2   url                              833 non-null    object 
 3   thumbnail                        833 non-null    object 
 4   lat                              831 non-null    float64
 5   lng                              831 non-null    float64
 6   rating_overall                   467 non-null    float64
 7   reviews_count                    467 non-null    float64
 8   rating_accuracy                  467 non-null    float64
 9   rating_cleanliness               467 non-null    float64
 10  rating_value                     467 non-null    float64
 11  rating_location                  467 non-null    float64
 12  description           

### Checking duplicates

In [167]:
df.duplicated().sum()

np.int64(0)

### Checking Nulls count

In [168]:
df.isna().sum()

id                                   0
title                               50
url                                  0
thumbnail                            0
lat                                  2
lng                                  2
rating_overall                     366
reviews_count                      366
rating_accuracy                    366
rating_cleanliness                 366
rating_value                       366
rating_location                    366
description                          0
price_breakdown_baseprice_price      0
nightly_price                        0
price_price                          0
checkin_year                         0
checkin_month                        0
checkin_day                          0
checkin_weekday                      0
checkout_year                        0
checkout_month                       0
checkout_day                         0
checkout_weekday                     0
dtype: int64

### Add `has_rating` flag according to rating
Keep track of the replaced nulls

In [169]:
df['has_rating'] = df['rating_overall'].notna()

### Nulls Handling

In [170]:
# title
df['title'] = df['title'].fillna('Unknown')

# reviews
df['reviews_count'] = df['reviews_count'].fillna(0)

# drop rows without location
df = df.dropna(subset=['lat', 'lng']).copy()

# ratings
rating_cols = [
    'rating_overall',
    'rating_accuracy',
    'rating_cleanliness',
    'rating_value',
    'rating_location'
]

for col in rating_cols:
    df[col] = df[col].fillna(df[col].mean())

# check
print(df[rating_cols].isna().sum())

rating_overall        0
rating_accuracy       0
rating_cleanliness    0
rating_value          0
rating_location       0
dtype: int64


In [171]:
print(df.isna().sum())
df.head()

id                                 0
title                              0
url                                0
thumbnail                          0
lat                                0
lng                                0
rating_overall                     0
reviews_count                      0
rating_accuracy                    0
rating_cleanliness                 0
rating_value                       0
rating_location                    0
description                        0
price_breakdown_baseprice_price    0
nightly_price                      0
price_price                        0
checkin_year                       0
checkin_month                      0
checkin_day                        0
checkin_weekday                    0
checkout_year                      0
checkout_month                     0
checkout_day                       0
checkout_weekday                   0
has_rating                         0
dtype: int64


,id,title,url,thumbnail,lat,lng,rating_overall,reviews_count,rating_accuracy,rating_cleanliness,...,price_price,checkin_year,checkin_month,checkin_day,checkin_weekday,checkout_year,checkout_month,checkout_day,checkout_weekday,has_rating
0,1292713234154945394,"ETERNA.Suite W Jaccuzi, Pyramids View & Balcony",https://www.airbnb.com/rooms/12927132341549453...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.973873,31.146603,4.95,133.0,4.96,4.92,...,675,2026,5,31,6,2026,6,5,4,True
1,1508718511630646313,king khufu suite,https://www.airbnb.com/rooms/15087185116306463...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.986300,31.143100,5.00,62.0,5.00,5.00,...,527,2026,5,24,6,2026,5,29,4,True
2,1297327219631789358,Akasia Pyramids View,https://www.airbnb.com/rooms/12973272196317893...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.978100,31.145400,4.91,176.0,4.91,4.91,...,194,2026,6,24,2,2026,6,29,0,True
3,1606001853199411128,Mountain Cave | Pyramids View & Jacuzzi,https://www.airbnb.com/rooms/16060018531994111...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.979090,31.146920,4.92,12.0,5.00,4.75,...,593,2026,5,11,0,2026,5,16,5,True
4,1314833467096489875,Heaven of Pyramids,https://www.airbnb.com/rooms/13148334670964898...,https://a0.muscache.com/im/pictures/miso/Hosti...,29.978787,31.144191,4.71,125.0,4.74,4.61,...,117,2026,5,3,6,2026,5,8,4,True


#### convert the strings to float

In [172]:
cols = [
    'rating_overall',
    'reviews_count',
    'rating_accuracy',
    'rating_cleanliness',
    'rating_value',
    'rating_location',
    'id',
    'price_price'
]

for col in cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [173]:
# Remove $
df['price_breakdown_baseprice_price'] = df['price_breakdown_baseprice_price'].replace('[^0-9.]', '', regex=True)
df['price_breakdown_baseprice_price'] = pd.to_numeric(df['price_breakdown_baseprice_price'], errors='coerce')

In [174]:

df['price_breakdown_baseprice_price'].head()

0    750.00
1    526.87
2    193.18
3    592.44
4    116.10
Name: price_breakdown_baseprice_price, dtype: float64

## Further Cleaning 
(After scrabbing bedrooms and bathrooms data and saving to `raw/airbnb_rooms.csv`)

In [175]:
df2 = pd.read_csv('../Data/raw/airbnb_rooms.csv')
df2.head()

,id,bedrooms,bathrooms
0,1292713234154945394,1,1.0
1,1508718511630646313,1,NaN
2,1297327219631789358,1,1.0
3,1606001853199411128,1,1.0
4,1314833467096489875,1,NaN


In [176]:
df2.isnull().sum()

id             0
bedrooms      45
bathrooms    108
dtype: int64

In [177]:
df2['bathrooms'] = df2['bathrooms'].fillna(df2['bathrooms'].mode()[0])
df2['bedrooms'] = df2['bedrooms'].fillna(df2['bedrooms'].mode()[0])
df2.isnull().sum()

id           0
bedrooms     0
bathrooms    0
dtype: int64

In [178]:
df2['bedrooms'] = df2['bedrooms'].replace('0 (studio)', 0)
df2['bedrooms'] = pd.to_numeric(df2['bedrooms'], errors='coerce')

In [179]:
df = df.merge(df2, on="id", how="inner")  

In [180]:
df.head()

,id,title,url,thumbnail,lat,lng,rating_overall,reviews_count,rating_accuracy,rating_cleanliness,...,checkin_month,checkin_day,checkin_weekday,checkout_year,checkout_month,checkout_day,checkout_weekday,has_rating,bedrooms,bathrooms
0,1292713234154945394,"ETERNA.Suite W Jaccuzi, Pyramids View & Balcony",https://www.airbnb.com/rooms/12927132341549453...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.973873,31.146603,4.95,133.0,4.96,4.92,...,5,31,6,2026,6,5,4,True,1,1.0
1,1508718511630646313,king khufu suite,https://www.airbnb.com/rooms/15087185116306463...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.986300,31.143100,5.00,62.0,5.00,5.00,...,5,24,6,2026,5,29,4,True,1,1.0
2,1297327219631789358,Akasia Pyramids View,https://www.airbnb.com/rooms/12973272196317893...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.978100,31.145400,4.91,176.0,4.91,4.91,...,6,24,2,2026,6,29,0,True,1,1.0
3,1606001853199411128,Mountain Cave | Pyramids View & Jacuzzi,https://www.airbnb.com/rooms/16060018531994111...,https://a0.muscache.com/im/pictures/hosting/Ho...,29.979090,31.146920,4.92,12.0,5.00,4.75,...,5,11,0,2026,5,16,5,True,1,1.0
4,1314833467096489875,Heaven of Pyramids,https://www.airbnb.com/rooms/13148334670964898...,https://a0.muscache.com/im/pictures/miso/Hosti...,29.978787,31.144191,4.71,125.0,4.74,4.61,...,5,3,6,2026,5,8,4,True,1,1.0


In [181]:
df.to_csv('../Data/raw/airbnb_processed.csv',index=False, encoding="utf-8-sig")